# Libraries

In [ ]:
import numpy as np
import pandas as pd
import time

import seaborn as sns
import tensorflow as tf
import random
import os

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt

In [ ]:
print("GPU available:", tf.config.list_physical_devices('GPU'))

# Fix the seed

In [ ]:
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

# Dataset



---



**1.   Data reading**

In [ ]:
df = pd.read_csv('Timeseries_32.207_-6.532_SA3_300kWp_crystSi_18_33deg_0deg_2010_2023.csv',
                 skiprows=10,
                 nrows=122723)
print(f"Table shape: {df.shape}")
print(df.head())
# print("Detected columns:", df.columns.tolist())
# print(df.head())



---





2.   **Feature Engineering**



*   Temporal cycling







In [ ]:
# Convert to datetime format
df['time'] = pd.to_datetime(df['time'], format='%Y%m%d:%H%M')
df['hour'] = df['time'].dt.hour  # Extract hour (0 to 23)
df['month'] = df['time'].dt.month  # Extract month (1 to 12) for seasonality
# Select columns for the model
features = ['P', 'Gb(i)', 'Gd(i)', 'T2m', 'WS10m', 'hour', 'month']
data = df[features].values
# Hour Cycling (24-hour period)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
# Month Cycling (12-month period)
df['month_sin'] = np.sin(2 * np.pi * (df['month']-1) / 12)
df['month_cos'] = np.cos(2 * np.pi * (df['month']-1) / 12)

# We define the columns that we want to keep for the model
# Note: We keep Gb(i) and Gd(i) because they are crucial for the solar
features_finales = [
    'P', 'Gb(i)', 'Gd(i)', 'T2m', 'WS10m',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos'
]

# We create the working dataset
df_final = df[features_finales]

print("Final dataset overview (9 columns):")
print(df_final.head())



*   Visualization


In [ ]:
# We trace hour_sin and hour_cos over the first 48 hours
plt.figure(figsize=(12, 4))
plt.plot(df['hour_sin'][:48], label='Hour Sine')
plt.plot(df['hour_cos'][:48], label='Hour Cosine')
plt.title("Temporal cycling verification (48h)")
plt.legend()
plt.show()



*   Standardization



In [ ]:
scaler = MinMaxScaler(feature_range=(-1, 1))
data_scaled = scaler.fit_transform(df_final)



*   Creation of Matrices (3D)


In [ ]:
def create_sequences(data, window_size):
    X = []
    y = []

    # Iterate through the dataset
    for i in range(len(data) - window_size):
        # The X sequence: the past 12 hours (all 9 columns)
        # From index i up to i + window_size (exclusive)
        X.append(data[i : i + window_size, :])

        # The Target y: Power P (column 0) at time t + window_size
        # This is exactly the value that follows the sequence
        y.append(data[i + window_size, 0])

    return np.array(X), np.array(y)

# Application to your normalized Afourer data
X, y = create_sequences(data_scaled, window_size=24)



*   Slicing




In [ ]:
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# we’re taking 2023 for the test
X_test = X_test[-8760:]
y_test = y_test[-8760:]

y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

print(f"Ready for training! X_Train size: {X_train.shape}")
print(f"Ready for training! y_Train size: {y_train.shape}")
print(f"Ready for testing! X_test size: {X_test.shape}")
print(f"Ready for testing! y_test size: {y_test.shape}")

# CNN Model



---





**1.   Architecture**




---



In [ ]:
model_cnn = Sequential([
    # Filters: 64, Kernel size: 3 (looks at 3 hours by 3 hours)
    Conv1D(filters=64, kernel_size=3, padding="causal", activation='relu', input_shape=(24, 9)),

    # MaxPooling: Reduces dimension by keeping the most important features
    MaxPooling1D(pool_size=2),

    # Second layer to capture more complex patterns
    Conv1D(filters=32, kernel_size=3, padding="causal", activation='relu'),

    Flatten(),

    Dense(50, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

model_cnn.compile(optimizer='adam', loss='mse', metrics=['mae'])
model_cnn.summary()






**2.  Training**





In [ ]:
start_time = time.time()

history_cnn = model_cnn.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)

end_time = time.time()
print(f"--- Total CNN training time: {end_time - start_time:.2f} seconds ---")



**3.  Visualization**



In [ ]:
def plot_learning_curves(history):
    # Retrieve history data
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(loss) + 1)

    # Create the graph
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, loss, 'bo-', label='Training Error (Loss)')
    plt.plot(epochs, val_loss, 'ro-', label='Validation Error (Val_loss)')

    plt.title('Learning Curves - LSTM Model (Afourer)')
    plt.xlabel('Epochs')
    plt.ylabel('Error (MSE)')
    plt.legend()
    plt.grid(True)
    plt.show()

    return loss, val_loss

In [ ]:
loss, val_loss = plot_learning_curves(history_cnn)

In [ ]:
loss, val_loss

In [ ]:
y_pred_scaled = model_cnn.predict(X_test)
# Denormalization function (Dummy Matrix Method)
def denormalize(scaled_values, scaler, n_features=9):
    # Create an empty matrix with 9 columns (the number of original columns)
    dummy = np.zeros((len(scaled_values), n_features))
    # Place our (scaled) values in the first column (Power P)
    dummy[:, 0] = scaled_values.flatten()
    # Apply the inverse of MinMaxScaler
    inverse = scaler.inverse_transform(dummy)
    # Only retrieve the first column (the real power)
    return inverse[:, 0]

# Convert to real units (Watts according to my dataset)
y_test_real = denormalize(y_test, scaler)
y_pred_real = denormalize(y_pred_scaled, scaler)
y_pred_real = np.maximum(y_pred_real, 0)

# Calculate scientific scores
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
mae = mean_absolute_error(y_test_real, y_pred_real)
r2 = r2_score(y_test_real, y_pred_real)
nMAE = 100 * (mae / np.max(y_test_real))
nRMSE  = 100 * (rmse / np.max(y_test_real))

print(f"--- RESULTS (Window = 24h) ---")
print(f"RMSE : {rmse:.3f}")
print(f"MAE  : {mae:.3f}")
print(f"nRMSE : {nRMSE:.3f}","%")
print(f"nMAE  : {nMAE:.3f}","%")
print(f"R²   : {r2:.4f}")

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(y_test_real[200:368], label='Real Afourer', color='red', alpha=0.6)
plt.plot(y_pred_real[200:368], label='LSTM Prediction', color='blue', linestyle='--')
plt.ylabel('Power (Watt)')
plt.title('Zoom in a week of Test ')
plt.legend()
plt.show()

In [ ]:
y_pred_real[200:368]